## Publications

In [56]:
from discovery_child_development.utils.jsonl_utils import load_jsonl
import pandas as pd
import json
import altair as alt

from discovery_child_development import PROJECT_DIR, S3_BUCKET
from discovery_child_development.utils import plotting_utils as pu

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'



In [57]:
relevant_df = pd.read_csv(ENRICHED_DATA_DIR / 'relevant_labelled_df.csv')

In [ ]:
pubs_metadata = "publications_metadata.jsonl"
metadata_dict = load_jsonl(pubs_metadata)

In [105]:
pubs_totals = (
    pd.DataFrame(json.load(open("total_publications_per_country.json", "r")))
    .assign(fraction = lambda df: df['total_publications'] / df['total_publications'].sum())
)
pubs_totals

,country_code,total_publications,fraction
0,AD,385,0.000006
1,AE,104268,0.001651
2,AF,5359,0.000085
3,AL,13335,0.000211
4,AM,17560,0.000278
...,...,...,...
175,XK,8636,0.000137
176,YE,17385,0.000275
177,ZA,363382,0.005754
178,ZM,14568,0.000231


In [52]:
metadata_dict[0].keys()

dict_keys(['id', 'doi', 'title', 'display_name', 'publication_year', 'publication_date', 'ids', 'language', 'primary_location', 'type', 'type_crossref', 'indexed_in', 'open_access', 'authorships', 'countries_distinct_count', 'institutions_distinct_count', 'corresponding_author_ids', 'corresponding_institution_ids', 'apc_list', 'apc_paid', 'has_fulltext', 'fulltext_origin', 'cited_by_count', 'cited_by_percentile_year', 'biblio', 'is_retracted', 'is_paratext', 'primary_topic', 'topics', 'keywords', 'concepts', 'mesh', 'locations_count', 'locations', 'best_oa_location', 'sustainable_development_goals', 'grants', 'datasets', 'versions', 'referenced_works_count', 'referenced_works', 'related_works', 'ngrams_url', 'abstract_inverted_index', 'cited_by_api_url', 'counts_by_year', 'updated_date', 'created_date'])

In [69]:
metadata_dict[0]['cited_by_count']

610

In [72]:
pub_countries = []
citations = []
for pub in metadata_dict:
    # Get the countries of the authors
    pub_country = []
    for author in pub['authorships']:
        for institution in author['institutions']:
            pub_country.append(institution['country_code'])
    pub_countries.append(set(pub_country))

    # Get the number of citations
    citations.append(pub['cited_by_count'])

ids = [pub['id'].split("/")[-1] for pub in metadata_dict]

In [80]:
metadata_df = (
    pd.DataFrame({'id': ids, 'country_code': pub_countries, 'cited_by_count': citations})
    .assign(country_code = lambda x: x['country_code'].apply(lambda y: list(y)))
)

In [122]:
metadata_df.to_csv(ENRICHED_DATA_DIR / 'pubs_metadata_df.csv', index=False)

In [82]:
relevant_meta_df = (
    relevant_df
    .query("Dataset == 'Publications'")
    .drop(columns=['country_code'])
    .merge(metadata_df, on='id', how='left')
)

In [112]:
pubs_counts = (
    relevant_meta_df
    .explode('country_code')
    .groupby('country_code')
    .agg(counts=("id", "count"))
    .reset_index()
    .assign(fraction = lambda df: df['counts'] / df['counts'].sum())
)

pubs_comparison = (
    pubs_counts
    .merge(pubs_totals, on='country_code', suffixes=('_data', '_totals'))
    .assign(fraction_diff = lambda df: df['fraction_data']/df['fraction_totals'])
    .sort_values('counts', ascending=False)
)

In [109]:
patent_comparison = pd.read_csv("global_vs_data_patents.csv")

In [117]:
pubs_comparison.round(3).head(10)

,country_code,counts,fraction_data,total_publications,fraction_totals,fraction_diff
168,US,10474,0.280,11459361,0.181,1.542
9,AU,2949,0.079,1419282,0.022,3.506
58,GB,2942,0.079,3128224,0.050,1.587
75,ID,1994,0.053,1954113,0.031,1.722
27,CA,1913,0.051,1641694,0.026,1.966
34,CN,1115,0.030,7827942,0.124,0.240
42,DE,883,0.024,2804105,0.044,0.531
124,NL,830,0.022,904640,0.014,1.548
22,BR,760,0.020,2057639,0.033,0.623
147,SE,710,0.019,583478,0.009,2.053


In [120]:
patent_comparison.head(15).round(3)

,country_code,counts_patents,fraction_patents,counts_totals,fraction_totals,fraction_diff
0,CN,8097,0.707,37560670,0.549,1.289
1,KR,957,0.084,3127339,0.046,1.830
2,US,661,0.058,8126884,0.119,0.486
3,JP,324,0.028,5736949,0.084,0.338
4,WO,312,0.027,2756940,0.040,0.677
5,RU,237,0.021,633492,0.009,2.237
6,EP,110,0.010,3549490,0.052,0.185
7,TW,105,0.009,1207724,0.018,0.520
8,AU,90,0.008,620563,0.009,0.867
9,BR,83,0.007,505183,0.007,0.982
